# 13 — Frozen classifier: final chronological holdout

**One-time final evaluation.** Notebook 12 showed tuned `C=0.03` and fixed `C=0.10` are effectively tied. We freeze **Logistic L2, C=0.10**, original return-tertile target, historical/peer-market features, and 5- plus 20-bar realized volatility (exclude 10-bar). The newest 20% of global timestamps have not been used for development model selection.

**Do not use these final holdout results to choose new features or tune the model.** If performance is disappointing, document it as the final result for this development cycle and establish a new future test period before further tuning.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src'/'stock_ml_dynamic').exists():
    raise RuntimeError('Run this notebook from the stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.final_holdout import fit_and_score_holdout, plot_holdout
candidates=[ROOT/'artifacts'/'master_tables'/'1d', ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
master_dir=next((p for p in candidates if (p/'master_labeled.parquet').exists()),None)
if master_dir is None: raise FileNotFoundError('Master table not found. Run Notebook 03 first.')
master, latest, bundle, manifest=load_master(master_dir)
print('Master:',master_dir,'rows:',len(master),'symbols:',master.symbol.nunique())

## Freeze and evaluate once

This operation refits all learned transforms on the development period, uses at most 120,000 training rows (as in Notebook 12), and scores **every eligible labeled holdout row**. The final output directory has a no-overwrite guard to discourage repeated holdout-driven tuning.

In [ ]:
OUTPUT=ROOT/'artifacts'/'final_holdout'/'1d'
scores,predictions,roc,confusion,eligibility=fit_and_score_holdout(
    master, holdout_fraction=0.20, max_train_rows=120_000,
    selected_k=4, cluster_fit_limit=15_000,
    random_state=42, output_dir=OUTPUT)
display(scores)
print('Holdout symbols:',predictions.symbol.nunique(), 'Rows:',len(predictions))
print('Excluded tickers:',eligibility.loc[~eligibility.eligible,'symbol'].tolist() if 'eligible' in eligibility else 'See universe_eligibility.csv')

## Accuracy, discrimination and error analysis

Compare the frozen model against its prior-class baseline. Macro one-vs-rest AUC is not the same as directional profitability. Check class-level confusion and the class proportions before interpreting accuracy.

In [ ]:
display(confusion)
print('Actual class proportions:')
display(predictions.actual_direction.value_counts(normalize=True).sort_index().rename('share'))
plot_holdout(scores,roc,confusion,OUTPUT)
print('Saved to:',OUTPUT)
print('Holdout status: EVALUATED — do not retune on this period.')

## Interpretation checklist

- Does the frozen classifier beat the prior baseline on accuracy, balanced accuracy, and macro AUC?
- Compare the **single holdout** result to Notebook 12's five-fold development averages (39.38% accuracy; 0.5566 macro AUC for fixed L2).
- Review class prevalence, confusion matrix, and timing before considering deployment.
- **Do not** iterate on the holdout after seeing results. For new research, establish a new prospective test period and log a fresh hypothesis.

Artifacts: `artifacts/final_holdout/1d/final_holdout_metrics.csv`, `final_holdout_predictions.parquet`, `final_holdout_roc.parquet`, `final_holdout_confusion.csv`, `final_holdout_manifest.json`, and plots.